In [1]:
import subprocess
import os

repo_url = "https://github.com/ldklab/scored23_release.git"
target_dir = "scored23_release"

# Clone repository
if not os.path.exists(target_dir):
    subprocess.run(
        ["git", "clone", repo_url],
        check=True
    )
else:
    print("Repository already exists")

# Dataset path
dataset_path = os.path.join(target_dir, "ASTanalysis", "Dataset")
print("Dataset location:", dataset_path)


Repository already exists
Dataset location: scored23_release\ASTanalysis\Dataset


In [65]:
import os

class CodeDatasetLoader:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path

    def load(self):
        samples = []
        label_map = {
            "CONTROL": 0,     # HUMAN
            "AUTOPOLI": 1     # AI
        }

        for group, label in label_map.items():
            group_path = os.path.join(self.dataset_path, group)

            if not os.path.exists(group_path):
                raise FileNotFoundError(f"Missing folder: {group_path}")

            for author in os.listdir(group_path):
                author_path = os.path.join(group_path, author)

                if not os.path.isdir(author_path):
                    continue

                for file in os.listdir(author_path):
                    if not file.endswith(".c"):
                        continue

                    file_path = os.path.join(author_path, file)

                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        samples.append({
                            "code": f.read(),
                            "label": label,
                            "author": author,
                            "group": group,
                            "file": file,
                            "path": file_path
                        })

        return samples

In [67]:
DATASET_PATH = r"C:\Users\Anota\ASE\scored23_release\ASTanalysis\Dataset"
import os

print("Exists?", os.path.exists(DATASET_PATH))
print("Contents:", os.listdir(DATASET_PATH))


Exists? True
Contents: ['.DS_Store', 'Autopilot', 'Control']


In [69]:
import os

class CodeDatasetLoader:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path

    def load(self):
        samples = []

        # Map folder names to class labels
        # Control    -> Human-written code (0)
        # Autopilot  -> AI-generated code (1)
        label_map = {
            "control": 0,
            "autopilot": 1
        }

        for folder in os.listdir(self.dataset_path):

            # Ignore hidden system files such as .DS_Store
            if folder.startswith("."):
                continue

            key = folder.lower()
            if key not in label_map:
                continue

            label = label_map[key]
            group_path = os.path.join(self.dataset_path, folder)

            if not os.path.isdir(group_path):
                continue

            for author in os.listdir(group_path):

                # Ignore hidden folders
                if author.startswith("."):
                    continue

                author_path = os.path.join(group_path, author)
                if not os.path.isdir(author_path):
                    continue

                for file in os.listdir(author_path):

                    # Only process C source files and ignore hidden files
                    if file.startswith(".") or not file.endswith(".c"):
                        continue

                    file_path = os.path.join(author_path, file)

                    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                        samples.append({
                            "code": f.read(),
                            "label": label,
                            "author": author,
                            "group": folder,
                            "file": file,
                            "path": file_path
                        })

        return samples


In [71]:
loader = CodeDatasetLoader(DATASET_PATH)
dataset = loader.load()

print("Total samples:", len(dataset))

Total samples: 508


In [72]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head(508)


,code,label,author,group,file,path
0,"int list_add_item_at_pos(node **head, char *it...",1,Machine1,Autopilot,addItem_Machine1.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
1,"int list_cost_sum(node *head, float *total) {\...",1,Machine1,Autopilot,costSum_Machine1.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
2,int list_find_highest_price_item_position(node...,1,Machine1,Autopilot,highestPrice_Machine1.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
3,"int list_item_to_string(node *head, char *str)...",1,Machine1,Autopilot,itemToString_Machine1.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
4,int list_init(node **head) {\n *head = NULL...,1,Machine1,Autopilot,listInitialization_Machine1.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
...,...,...,...,...,...,...
503,int list_print(node *head) {\n // TODO: Imp...,0,Human9,Control,printList_Human9.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
504,"int list_remove_item_at_pos(node **head, int p...",0,Human9,Control,removeItem_Human9.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
505,"int list_save(node *head, char *filename)\n{\n...",0,Human9,Control,save_Human9.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...
506,"int list_swap_item_positions(node **head, int ...",0,Human9,Control,swapItem_Human9.c,C:\Users\Anota\ASE\scored23_release\ASTanalysi...


In [73]:
df["label"].value_counts()
count_0 = (df["label"] == 0).sum()
count_1 = (df["label"] == 1).sum()

print("Label 0 (Human):", count_0)
print("Label 1 (AI):", count_1)


Label 0 (Human): 181
Label 1 (AI): 327


In [101]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
import numpy as np

# -----------------------------------
# Raw data
# -----------------------------------

X = df["code"].values
y = df["label"].values   # 0 = Human, 1 = AI

# -----------------------------------
# Character-level TF-IDF (surface style)
# -----------------------------------

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 6),      # wider range
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    norm="l2"
)

# -----------------------------------
# Character within word boundaries
# -----------------------------------

char_wb_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    norm="l2"
)

# -----------------------------------
# Word-level TF-IDF (structure)
# -----------------------------------

word_tfidf = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"\w+",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm="l2"
)

# -----------------------------------
# Combine all representations
# -----------------------------------

vectorizer = FeatureUnion([
    ("char", char_tfidf),
    ("char_wb", char_wb_tfidf),
    ("word", word_tfidf)
])

X_tfidf = vectorizer.fit_transform(X)

print("Hybrid TF-IDF shape:", X_tfidf.shape)


Hybrid TF-IDF shape: (508, 33614)


In [103]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    SGDClassifier
)
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestCentroid

# --------------------------------------------------
# Models to evaluate
# --------------------------------------------------

models = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000),
    "Passive Aggressive (Aggressive)": PassiveAggressiveClassifier(max_iter=1000, C=0.5),
    "Linear SVM": LinearSVC(class_weight="balanced"),
    "SGD (Modified Huber)": SGDClassifier(loss="modified_huber", class_weight="balanced"),
    "SGD (Perceptron)": SGDClassifier(loss="perceptron", class_weight="balanced"),
    "Elastic Net Logistic Regression": LogisticRegression(
        max_iter=3000,
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ),
    "Nearest Centroid": NearestCentroid(),
    "SGD (Hinge)": SGDClassifier(loss="hinge", class_weight="balanced")
}

# --------------------------------------------------
# Train / Test split INSIDE training code
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# --------------------------------------------------
# Training and evaluation
# --------------------------------------------------

results = []

for model_name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    report = classification_report(
        y_test,
        y_pred,
        target_names=["Human", "AI"],
        output_dict=True
    )
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        
        "Precision_Human": report["Human"]["precision"],
        "Recall_Human": report["Human"]["recall"],
        "F1_Human": report["Human"]["f1-score"],
        
        "Precision_AI": report["AI"]["precision"],
        "Recall_AI": report["AI"]["recall"],
        "F1_AI": report["AI"]["f1-score"],
        
        "Macro_F1": report["macro avg"]["f1-score"],
        "Weighted_F1": report["weighted avg"]["f1-score"]
    })

results_df = pd.DataFrame(results).sort_values(
    by="Weighted_F1", ascending=False
)

results_df

Training samples: 355
Testing samples: 153


C:\Users\Anota\anaconda3\Lib\site-packages\sklearn\neighbors\_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(


,Model,Accuracy,Precision_Human,Recall_Human,F1_Human,Precision_AI,Recall_AI,F1_AI,Macro_F1,Weighted_F1
2,Linear SVM,0.993464,1.000000,0.981818,0.990826,0.989899,1.000000,0.994924,0.992875,0.993451
8,SGD (Hinge),0.986928,1.000000,0.963636,0.981481,0.980000,1.000000,0.989899,0.985690,0.986873
0,Passive Aggressive,0.980392,0.981481,0.963636,0.972477,0.979798,0.989796,0.984772,0.978624,0.980352
3,SGD (Modified Huber),0.973856,0.947368,0.981818,0.964286,0.989583,0.969388,0.979381,0.971834,0.973955
1,Passive Aggressive (Aggressive),0.973856,0.981132,0.945455,0.962963,0.970000,0.989796,0.979798,0.971380,0.973746
6,Logistic Regression,0.973856,1.000000,0.927273,0.962264,0.960784,1.000000,0.980000,0.971132,0.973624
4,SGD (Perceptron),0.960784,0.915254,0.981818,0.947368,0.989362,0.948980,0.968750,0.958059,0.961064
5,Elastic Net Logistic Regression,0.921569,1.000000,0.781818,0.877551,0.890909,1.000000,0.942308,0.909929,0.919029
7,Nearest Centroid,0.875817,0.846154,0.800000,0.822430,0.891089,0.918367,0.904523,0.863476,0.875012


In [119]:
import joblib

# --------------------------------------------------
# Select best model based on Weighted_F1
# --------------------------------------------------

best_model_name = results_df.iloc[0]["Model"]
print("Best model selected:", best_model_name)

best_model = models[best_model_name]

# --------------------------------------------------
# Retrain best model on training data
# --------------------------------------------------

best_model.fit(X_train, y_train)

# --------------------------------------------------
# Save model and vectorizer
# --------------------------------------------------

joblib.dump(best_model, "best_ai_code_classifier.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Best model and TF-IDF vectorizer saved successfully.")


Best model selected: Logistic Regression
Best model and TF-IDF vectorizer saved successfully.


In [123]:
# --------------------------------------------------
# Recreate train/test split using raw source code
# --------------------------------------------------

X_raw = df["code"].values
y = df["label"].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples (raw):", len(X_train_raw))
print("Testing samples (raw):", len(X_test_raw))


Training samples (raw): 355
Testing samples (raw): 153


In [125]:

from sklearn.pipeline import Pipeline
import joblib

# --------------------------------------------------
# Select best model based on Weighted_F1
# --------------------------------------------------

best_model_name = results_df.iloc[0]["Model"]
print("Best model selected:", best_model_name)

best_classifier = models[best_model_name]

# --------------------------------------------------
# Build full pipeline (TF-IDF + classifier)
# --------------------------------------------------

pipeline = Pipeline([
    ("tfidf", tfidf),        # Same TF-IDF configuration
    ("clf", best_classifier)
])

# --------------------------------------------------
# Train pipeline on raw training data
# --------------------------------------------------

pipeline.fit(X_train_raw, y_train)

# --------------------------------------------------
# Save pipeline
# --------------------------------------------------

joblib.dump(pipeline, "best_ai_code_pipeline.pkl")

print("Pipeline saved successfully.")


Best model selected: Logistic Regression
Pipeline saved successfully.


In [153]:
import joblib

pipeline = joblib.load("best_ai_code_pipeline.pkl")

new_code = ["int main()  { return 0;}"]

prediction = pipeline.predict(new_code)
label = "AI" if prediction[0] == 1 else "Human"

print("Prediction:", label)



Prediction: AI


In [157]:
import os
os.getcwd()

'C:\\Users\\Anota\\ASE'

In [109]:
# ==================================================
# Enhanced Pseudo-AST Feature Extraction (Improved)
# 100% requirements-safe
# ==================================================

import pandas as pd
import re
from collections import Counter

CONTROL_KEYWORDS = [
    "if", "else", "for", "while", "do", "switch",
    "case", "default", "break", "continue", "return"
]

TYPE_KEYWORDS = [
    "int", "float", "double", "char", "void",
    "long", "short", "unsigned", "signed", "struct"
]

def extract_enhanced_pseudo_ast(code: str):
    features = Counter()
    lines = code.splitlines()
    total_lines = max(len(lines), 1)

    # ---------- Control keywords ----------
    control_counts = {}
    for kw in CONTROL_KEYWORDS:
        c = len(re.findall(rf"\b{kw}\b", code))
        control_counts[kw] = c
        features[f"kw_{kw}"] = c

    total_control = sum(control_counts.values()) + 1

    # Ratios (important)
    for kw, c in control_counts.items():
        features[f"kw_{kw}_ratio"] = c / total_control

    # ---------- Types ----------
    for kw in TYPE_KEYWORDS:
        features[f"type_{kw}"] = len(re.findall(rf"\b{kw}\b", code))

    # ---------- Structural symbols ----------
    features["brace_open"] = code.count("{")
    features["paren_open"] = code.count("(")
    features["semicolon"] = code.count(";")

    # ---------- Nesting depth proxy ----------
    depth = 0
    max_depth = 0
    for ch in code:
        if ch == "{":
            depth += 1
            max_depth = max(max_depth, depth)
        elif ch == "}":
            depth = max(depth - 1, 0)

    features["max_nesting_depth"] = max_depth
    features["avg_nesting_depth"] = max_depth / (features["brace_open"] + 1)

    # ---------- Functions ----------
    funcs = re.findall(r"\w+\s+\w+\s*\([^)]*\)\s*{", code)
    num_funcs = max(len(funcs), 1)

    features["functions"] = num_funcs
    features["control_per_function"] = total_control / num_funcs
    features["lines_per_function"] = total_lines / num_funcs

    # ---------- Operators ----------
    op_count = len(re.findall(r"[+\-*/%=<>!]=?|&&|\|\|", code))
    features["operators"] = op_count
    features["operators_per_line"] = op_count / total_lines

    # ---------- Line statistics ----------
    empty_lines = sum(1 for l in lines if not l.strip())
    features["empty_line_ratio"] = empty_lines / total_lines

    return features

# --------------------------------------------------
# Apply to DataFrame
# --------------------------------------------------

ast_feature_dicts = [extract_enhanced_pseudo_ast(code) for code in df["code"]]

df_ast = pd.DataFrame(ast_feature_dicts).fillna(0)
df_ast["label"] = df["label"].values

print("Enhanced AST DataFrame shape:", df_ast.shape)
df_ast.head()


Enhanced C/C++ AST shape: (508, 65)


,kw_if,kw_else,kw_for,kw_while,kw_do,kw_switch,kw_case,kw_default,kw_break,kw_continue,...,semicolon,max_nesting_depth,avg_nesting_depth,functions,control_per_function,lines_per_function,operators,operators_per_line,empty_line_ratio,label
0,5,1,1,0,0,0,0,0,0,0,...,18,3,0.428571,4,3.250000,8.250000,61,1.848485,0.0,1
1,2,0,0,1,0,0,0,0,0,0,...,9,2,0.400000,3,2.333333,6.666667,46,2.300000,0.0,1
2,4,0,1,0,0,0,0,0,0,0,...,12,3,0.428571,3,3.000000,8.333333,38,1.520000,0.0,1
3,2,0,0,0,0,0,0,0,0,0,...,4,2,0.500000,2,3.000000,6.000000,21,1.750000,0.0,1
4,0,0,0,0,0,0,0,0,0,0,...,2,1,0.500000,1,2.000000,4.000000,4,1.000000,0.0,1


In [111]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from sklearn.linear_model import (
    LogisticRegression,
    PassiveAggressiveClassifier,
    SGDClassifier
)
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestCentroid

# --------------------------------------------------
# Read features from df_ast
# --------------------------------------------------

X = df_ast.drop(columns=["label"]).values
y = df_ast["label"].values

# --------------------------------------------------
# Models to evaluate
# --------------------------------------------------

models = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000),
    "Passive Aggressive (Aggressive)": PassiveAggressiveClassifier(max_iter=1000, C=0.5),
    "Linear SVM": LinearSVC(class_weight="balanced"),
    "SGD (Modified Huber)": SGDClassifier(loss="modified_huber", class_weight="balanced"),
    "SGD (Perceptron)": SGDClassifier(loss="perceptron", class_weight="balanced"),
    "Elastic Net Logistic Regression": LogisticRegression(
        max_iter=3000,
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        class_weight="balanced"
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    ),
    "Nearest Centroid": NearestCentroid(),
    "SGD (Hinge)": SGDClassifier(loss="hinge", class_weight="balanced")
}

# --------------------------------------------------
# Train / Test split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# --------------------------------------------------
# Training and evaluation
# --------------------------------------------------

results = []

for model_name, model in models.items():
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    report = classification_report(
        y_test,
        y_pred,
        target_names=["Human", "AI"],
        output_dict=True
    )
    
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        
        "Precision_Human": report["Human"]["precision"],
        "Recall_Human": report["Human"]["recall"],
        "F1_Human": report["Human"]["f1-score"],
        
        "Precision_AI": report["AI"]["precision"],
        "Recall_AI": report["AI"]["recall"],
        "F1_AI": report["AI"]["f1-score"],
        
        "Macro_F1": report["macro avg"]["f1-score"],
        "Weighted_F1": report["weighted avg"]["f1-score"]
    })

results_df = pd.DataFrame(results).sort_values(
    by="Weighted_F1", ascending=False
)

results_df

Training samples: 355
Testing samples: 153


C:\Users\Anota\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Anota\anaconda3\Lib\site-packages\sklearn\neighbors\_nearest_centroid.py:244: UserWarning: self.within_class_std_dev_ has at least 1 zero standard deviation.Inputs within the same classes for at least 1 feature are identical.
  warnings.warn(


,Model,Accuracy,Precision_Human,Recall_Human,F1_Human,Precision_AI,Recall_AI,F1_AI,Macro_F1,Weighted_F1
6,Logistic Regression,0.810458,0.691176,0.854545,0.764228,0.905882,0.785714,0.841530,0.802879,0.813742
2,Linear SVM,0.797386,0.676471,0.836364,0.747967,0.894118,0.775510,0.830601,0.789284,0.800896
5,Elastic Net Logistic Regression,0.790850,0.666667,0.836364,0.741935,0.892857,0.765306,0.824176,0.783056,0.794612
3,SGD (Modified Huber),0.718954,0.607143,0.618182,0.612613,0.783505,0.775510,0.779487,0.696050,0.719500
1,Passive Aggressive (Aggressive),0.712418,0.569620,0.818182,0.671642,0.864865,0.653061,0.744186,0.707914,0.718108
8,SGD (Hinge),0.712418,0.689655,0.363636,0.476190,0.717742,0.908163,0.801802,0.638996,0.684752
4,SGD (Perceptron),0.653595,0.510638,0.872727,0.644295,0.881356,0.530612,0.662420,0.653358,0.655905
7,Nearest Centroid,0.555556,0.415584,0.581818,0.484848,0.697368,0.540816,0.609195,0.547022,0.564496
0,Passive Aggressive,0.464052,0.398496,0.963636,0.563830,0.900000,0.183673,0.305085,0.434457,0.398098
